# Tuesday Challenge: creating the best ERK2 scoring function ‼️

## Challenge 2/2 - the ultimate challenge

**Author:** [Albert J. Kooistra](https://drug.ku.dk/staff/?pure=en/persons/612712), 2023-2025, University of Copenhagen

From a high throughput screening on ERK2, we have gathered a dataset of nearly 60 thousand compounds with experimentally determined inhibitory (in)activity on ERK2.

We removed 25% from this high throughput experimental dataset and made this the next challenge. Note that this is a actual real-world problem and this is a difficult one! So, you can expect very low enrichments - a gain in enrichment in any form is an achievement! Try different options / combinations to get to your ultimate scoring functions!

Each group has two submissions for each challenge and we will unveil the winning group(s) during the Feedback session next week 🏆

## Installation and import of libraries and functions

Simply execute the code cells below to get started.

In [ ]:
# Installing missing libraries necessary for processing the docking poses
!pip install -q rdkit prolif==1.1.0
!pip install -q py3Dmol

In [ ]:
# Import packages and libraries
import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np
import os
import pandas as pd
import prolif as plf
import py3Dmol
import re
import seaborn as sns

# Cherry-picking parts of packages
from ipywidgets import interact, fixed, IntSlider
from rdkit import Chem
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from prolif.plotting.network import LigNetwork
from rdkit import Geometry

# Disabling warnings (can be tricky)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# HELPER FUNCTIONS
# No need to read the code or interpret

# Hover functions for the molecular viewer (Javascript code)
hover_func = """
  function(atom,viewer) {
    if (!atom.label) 
      atom.label = viewer.addLabel(atom.resn + " " + atom.resi,
      {position: atom, backgroundColor: 'black', fontColor:'white'});
  }"""

# Unhover function for the molecular viewer (Javascript code)
unhover_func = """function(atom,viewer) {if (atom.label) { viewer.removeLabel(atom.label); delete atom.label; }}"""

# proLIF - redefining HB-acceptors (slightly wider angle)
class HBAcceptor(plf.interactions.HBAcceptor):
   def __init__(self): 
     super().__init__(angles=(120, 180))

# proLIF - redefining HB-donors (slightly wider angle)
class HBDonor(plf.interactions.HBDonor):
   def __init__(self): 
      super().__init__(angles=(120, 180))

# Functions based on TeachOpenCADD - T007
# https://projects.volkamerlab.org/teachopencadd/index.html

def plot_roc_curves_for_models(score_column, data, save_png = False, negative_scores = True):
    """
    Helper function to plot customized roc curve.

    Parameters
    ----------
    score_column: string
        Name of the column with the scores
    data: dataframe
        Dataframe with two columns: <Score> and "Active"
    save_png: bool
        Save image to disk (default = False)

    Returns
    -------
    fig:
        Figure.
    """

    
    if negative_scores:
      data = data.copy()
      data[score_column] = -1 * data[score_column]

    fig, ax = plt.subplots()
    fig.set_dpi(150)

    # Compute ROC plot with False postive rate and True positive rate
    fpr, tpr, thresholds = roc_curve(data["Active"], data[score_column], pos_label=1)

    # Calculate Area under the curve to display on the plot
    auc = roc_auc_score(data["Active"], data[score_column])

    # Plot the computed values
    ax.plot(fpr, tpr, label=(f"{score_column} AUROC = {auc:.2f}"))

    # Custom settings for the plot
    ax.plot([0, 1], [0, 1], "r--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Receiver Operating Characteristic")
    ax.legend(loc="lower right")

    # Save plot
    if save_png:
        fig.savefig(f"{DATA}/roc_auc", dpi=300, bbox_inches="tight", transparent=True)

    return fig

## Setting the stage: getting all data

In [ ]:
# Getting the full data package and extract
# Only do this if we don't already have it!
if not os.path.isfile("Week_5_Tuesday_dataset_chal2.tgz"):
    # Downloading the dataset
    ! wget -q --show-progress "https://www.dropbox.com/s/2kkaejai3f9myq3/Week_5_Tuesday_dataset_large.tgz?dl=0" -O Week_5_Tuesday_dataset_chal2.tgz
    # Extracting the dataset
    ! tar -xvzf Week_5_Tuesday_dataset_chal2.tgz 2>/dev/null

In [ ]:
%ls *

### Checking the data

In the main folder we have different PDB files:
* 3x ERK2 crystal structure in different conformations
* 1x AlphaFold model
* 1x the experimentally-determined binding mode of our mystery reference ligand E94
* 1x training_set.txt - the classification active/inactive for our training ligands
* 1x The data archive (the .tgz file)
* 1x this notebook

Then we have a compounds folder:
* training.smi - the molecular structure of the training compounds in SMILES format
* challenge.smi - the molecular structure of the challenge compounds in SMILES format

And finally, we have pre-docked all compounds, both from the training and challenge set, into all proteins. For each protein structure/model, we have one <code/name>\_docked folder. In each folder you will find:
* training_features.csv.gz - docking scores and features for the training compounds
* challenge_features.csv.gz - docking scores and features for the challenge compounds
* training_ifp.csv.gz - interaction fingerprints for the training compounds
* challenge_ifp.csv.gz - interaction fingerprints for the challenge compounds

‼️ Note: we obtained 10 docking poses for each compounds, but in a few cases we had multiple isomers of a compound resulting in more than 10 docking poses for the same compound.

Below a few examples of reading in the data. Note that you can borrow code from yesterdays notebook as well!

In [ ]:
# Reading in the docking scores and ifp from docking the 4FV7 protein structure
features = pd.read_csv("4fv7_docked/training_features.csv.gz")
print("=== FEATURES ===")
print(features.head())

ifp = pd.read_csv("4fv7_docked/training_ifp.csv.gz")
print("=== INTERACTIONS ===")
print(ifp.head())

# Reading in the labels/classes for the compounds in the training set
cpd_classes = pd.read_csv("training_set.txt", sep=" ", dtype=int, index_col=0)
print("=== LABELS/CLASSES ===")
cpd_classes.head()

Take a look at the memory usage indicator on top. By now we have already used quite a bit and haven't even gotten started. So just to clean up, to delete your (unused) variables again when you don't need them anymore. 

In [ ]:
del features
del ifp
del cpd_classes

## Building a scoring function

This is (of course) the key and fun part 😁\
However, keep in mind that machine learning alone is not the only way to go and you can incorporate knowledge you have gained from yesterday's analysis!

On top, we have quite some input to be used for our machine learning:
* Docking in 4 structures each yielding 27 features
* Interaction FingerPrints for each docking yielding ~140 features per docking structure
* At least 10 docking poses per compound to select from per docking structure

Follow roughly the following steps to create a scoring function:
*Training your model*:
* Select your data sources and read them
* Perform a feature selection - which features are you going to use as input for machine learning
* Merge the selected data sets
* MEMORY: if your memory usage is high, make a copy() of your dataframe and delete large variables
* Determine a strategy to perform the pose selection and reduce/clean your dataset (1 pose per compound)
* Add the activity class at the end of your dataset
* Test different classification methodologies - borrow from the code examples at the end of this notebook, from other notebooks, or use Google/ChatGPT
* Evaluate your models using ROC-curves and especially the early enrichment (EF1%)
* Repeat this until you are happy with your resulting model
* OPTIONAL: based on the available data, feel free to generate new features and add them to your dataset to enhance your model

*Applying your model*:
* Select the same data sources as used for training your model
* Merge them into a single dataframe in the same way (but this time without the activity class, since they are unknown)
* Apply your machine learning model to the challenge dataframe
* Generate a submission CSV containing the compound ID and the classification outcome
* Submit!

Use the code boxes below to generate your models and submission and don't forget to interpret the resulting scoring function!

# SCORING FUNCTION 1

In [ ]:
### YOUR CODE HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
### YOUR CODE HERE

YOUR ANSWER HERE

Submit your final two scoring lists and you are done!


# DONE 🎉🎉!